In [ ]:
import json
import logging
import os
import sys

from pathlib import Path

basepath = Path(
    r"C:\Users\s158699\Documents\PhD\Experiments\Simulation\aggregated_event_data"
)

sys.path.append(os.path.join(*basepath.parts))

from aggregated_event_data import simulate
from numpy import isclose
from pathlib import Path

logging.basicConfig(level=logging.INFO, force=True)
logger = logging.getLogger()

from aggregated_traces.utils.construct_ekg import (
    check_quantities,
    generate_networkx_di_graph,
    insert_DF_DP,
    insert_fractions,
    load_rdf_graph,
)
from aggregated_traces.utils.ekg_analysis import (
    compute_aggregation_uniformity,
    compute_aggregation_average_kl_trace_graph,
    compute_trace_probabilities,
    compute_number_of_merges_in_trace_graph,
    compute_number_of_steps_at_aggregations,
)

In [ ]:
from rdflib import Graph, URIRef, Variable
from typing import List


def select_packing_units_with_quality_issue(
    graph: Graph, threshold: float
) -> List[URIRef]:
    """
    Find packing unit(s) with average quality below a certain threshold.
    """

    query_lowest_quality = """
    PREFIX : <http://example.org/def/ekg/aggregated_traces/>
    SELECT DISTINCT
        ?packing_unit
        (sum(?device_quality)/count(?device_quality) as ?average_quality)
    WHERE {
        [] :bizStep "packing" ;
            :parentEntity ?packing_unit ;
            :device/:quality ?device_quality .

        ?packing_unit a :PackingUnit .
    }
    GROUP BY ?packing_unit
    """

    bindings = graph.query(query_lowest_quality).bindings

    return [
        b[Variable("packing_unit")]
        for b in bindings
        if b[Variable("average_quality")].toPython() < threshold
    ]


from pandas import DataFrame, Index, isna


def get_entity_record_number(df: DataFrame, entity: str) -> int:
    try:
        return Index(df["entity_target"]).get_loc(entity) + 1
    except KeyError:
        return None

# Process event logs

## Compute trace probabilities

In [ ]:
scenario = "scenario_15"
n_runs = 10

# Define the root cause entity
quality_threshold = 0.9

event_log_files = basepath.joinpath(f"logs/{scenario}/")
if not event_log_files.exists():
    os.mkdir(event_log_files)

result_files = Path(f"output/{scenario}/")
if not result_files.exists():
    os.mkdir(result_files)

for i in range(n_runs):
    # if i != 0:
    #     continue

    run_file_name = f"run_{i}.json"
    event_data_file = event_log_files.joinpath(run_file_name)
    simulate.main(
        config_file=basepath.joinpath(f"examples/notebook/scenarios/{scenario}.json"),
        runtime=1000,
        output_event_log_file=event_data_file,
    )

    print(event_data_file)

    # Parse data to RDF graph
    ekg_graph_rdf = load_rdf_graph(event_data_file)

    # Insert DF/DP relations
    ekg_graph_rdf = insert_DF_DP(ekg_graph_rdf)

    # Check incoming vs outgoing amount
    query_result_incoming_outgoing = check_quantities(ekg_graph_rdf)

    # Insert fractions on the relations
    ekg_graph_rdf = insert_fractions(ekg_graph_rdf)

    # Create NetworkX graph
    ekg_graph_nx = generate_networkx_di_graph(ekg_graph_rdf)

    # Find packing units with lowest (average) quality for which to find the root cause (entity)
    entities_source_backward = select_packing_units_with_quality_issue(
        graph=ekg_graph_rdf, threshold=quality_threshold
    )

    # Compute backward trace probabilities
    try:
        df_backward, edges_paths_backward = compute_trace_probabilities(
            rdf_trace_graph=ekg_graph_rdf,
            nx_trace_graph=ekg_graph_nx,
            source_entities=entities_source_backward,
            trace_backward=True,
        )
    except RuntimeError as e:
        logger.warning(e)
        continue

    # Compute statistics on trace graphs for source-target node pairs
    aggregation_kl = compute_aggregation_uniformity(graph=ekg_graph_rdf)
    for i in df_backward.index:
        # Compute number of merges in the trace graph
        df_backward.loc[i, "n_merges"] = compute_number_of_merges_in_trace_graph(
            trace_graph=ekg_graph_nx,
            source=df_backward.loc[i, "node_source"],
            target=df_backward.loc[i, "node_target"],
        )

        # Compute uniformity of splits and merges
        df_backward.loc[i, "split_merge_kl"] = (
            compute_aggregation_average_kl_trace_graph(
                trace_graph=ekg_graph_nx,
                aggregation_kl=aggregation_kl,
                source=df_backward.loc[i, "node_source"],
                target=df_backward.loc[i, "node_target"],
            )
        )

    # Get the number of steps executed at the aggregation events
    steps_aggregations = compute_number_of_steps_at_aggregations(
        graph=ekg_graph_rdf, events=df_backward["node_source"].unique()
    )
    df_backward["n_production_steps_aggregations"] = df_backward["node_source"].map(
        steps_aggregations
    )

    # Validate if probabilities add up to one
    if not all(
        isclose(
            df_backward.groupby(["entity_source", "product_model"])["probability"]
            .sum()
            .astype(float),
            1,
        )
    ):
        raise Warning("Sum of probabilities by production step do not add up to one!")

    df_backward["probability"] = df_backward["probability"].astype(float)
    with open(result_files.joinpath(run_file_name), "w") as f:
        json.dump(
            df_backward.to_dict(orient="records"),
            f,
        )

In [ ]:
ekg_graph_rdf.query("""
SELECT DISTINCT *
WHERE {
    [] :location ?location
}
""").bindings

## Compute steps to find root cause (entity)

In [ ]:
scenario = "scenario_14"

root_cause_entity = {
    "scenario_1": "http://example.org/id/ekg/aggregated_traces/DB2",
    "scenario_2": "http://example.org/id/ekg/aggregated_traces/DB2",
    "scenario_3": "http://example.org/id/ekg/aggregated_traces/DB2",
    "scenario_4": "http://example.org/id/ekg/aggregated_traces/DB1",
    "scenario_5": "http://example.org/id/ekg/aggregated_traces/DB1",
    "scenario_6": "http://example.org/id/ekg/aggregated_traces/DB1",
    "scenario_7": "http://example.org/id/ekg/aggregated_traces/DB1",
    "scenario_8": "http://example.org/id/ekg/aggregated_traces/WT0",
    "scenario_9": "http://example.org/id/ekg/aggregated_traces/WT0",
    "scenario_10": "http://example.org/id/ekg/aggregated_traces/WT0",
    "scenario_11": "http://example.org/id/ekg/aggregated_traces/WT0",
    "scenario_12": "http://example.org/id/ekg/aggregated_traces/WB1",
    "scenario_13": "http://example.org/id/ekg/aggregated_traces/WB1",
    "scenario_14": "http://example.org/id/ekg/aggregated_traces/WB1",
    "scenario_15": "http://example.org/id/ekg/aggregated_traces/WB1",
}[scenario]

probability_dicts = {}
for result_file in Path(f"output/{scenario}/").glob("*.json"):
    with open(result_file) as f:
        probability_dicts[result_file] = json.load(f)

### Include all entities

results = []
for result_file, probability_dict in probability_dicts.items():
    df_trace = DataFrame(probability_dict)

    # Aggregate to get probabilities for target entities
    df_trace_grouped = df_trace.groupby(["entity_source", "entity_target"]).agg(
        {
            "probability": "sum",
            "n_merges": "sum",
            "split_merge_kl": "mean",
            "n_production_steps_aggregations": list,
        }
    )
    df_trace_grouped.reset_index(inplace=True)

    entities_source = df_trace_grouped["entity_source"].unique()
    for entity_source in entities_source:
        df_selected = df_trace_grouped[
            df_trace_grouped["entity_source"] == entity_source
        ]

        # Shuffle/sample DataFrame for random order of inspection
        n_steps_random = get_entity_record_number(
            df_selected.sample(frac=1), root_cause_entity
        )

        # Sort values for informed order of inspection
        n_steps_sorted = get_entity_record_number(
            df_selected.sort_values("probability", ascending=False), root_cause_entity
        )

        results.append(
            {
                "file": result_file,
                "probabilities": df_selected.to_dict(orient="records"),
                "n_steps_random": n_steps_random,
                "n_steps_sorted": n_steps_sorted,
            }
        )

### Grouped by type (for example type of machine)

In [ ]:
group_key = "product_model"

results = []
for event_data_file, probability_dict in probability_dicts.items():
    df_trace = DataFrame(probability_dict)

    for group_value in df_trace[group_key].unique():
        # Aggregate to get probabilities for target entities
        df_trace_grouped = (
            df_trace[df_trace[group_key] == group_value]
            .groupby(["entity_source", "entity_target"])
            .agg(
                {
                    "probability": "sum",
                    "n_merges": "sum",
                    "split_merge_kl": "mean",
                    "n_production_steps_aggregations": list,
                }
            )
        )
        df_trace_grouped.reset_index(inplace=True)

        # Remove lists with NaN value
        df_trace_grouped["n_production_steps_aggregations"] = df_trace_grouped[
            "n_production_steps_aggregations"
        ].apply(lambda x: [] if any(not isinstance(i, list) for i in x) else x)

        entities_source = df_trace_grouped["entity_source"].unique()
        for entity_source in entities_source:
            df_selected = df_trace_grouped[
                df_trace_grouped["entity_source"] == entity_source
            ]

            # Shuffle/sample DataFrame for random order of inspection
            n_steps_random = get_entity_record_number(
                df_selected.sample(frac=1), root_cause_entity
            )

            # Sort values for informed order of inspection
            n_steps_sorted = get_entity_record_number(
                df_selected.sort_values("probability", ascending=False),
                root_cause_entity,
            )

            results.append(
                {
                    "file": event_data_file,
                    group_key: group_value,
                    "probabilities": df_selected.to_dict(orient="records"),
                    "n_steps_random": n_steps_random,
                    "n_steps_sorted": n_steps_sorted,
                }
            )

In [ ]:
df = DataFrame(results)
df.describe()

# Statistics

In [ ]:
from scipy.stats import entropy

## Uniformity of probabilities

In [ ]:
for i in df.index:
    df_group = DataFrame(df.loc[i, "probabilities"])

    n_targets = df_group["entity_target"].nunique()
    probabilities = df_group.groupby(["entity_source", "entity_target"])[
        "probability"
    ].sum()
    probabilities = probabilities.astype(float)

    kl_divergence = entropy(pk=probabilities.values, qk=[1 / n_targets] * n_targets)

    df.loc[i, "kl_divergence"] = (
        kl_divergence  # closer to 0 is closer to uniform distribution
    )

### Number of merges

In [ ]:
df["n_merges"] = df["probabilities"].apply(lambda x: sum(r["n_merges"] for r in x))

### Uniformity of merges and splits

In [ ]:
df["split_merge_kl"] = df["probabilities"].apply(
    lambda x: sum(r["split_merge_kl"] for r in x) / len(x)
)

### Average number of resources (per production step)

In [ ]:
with open(basepath.joinpath(f"examples/notebook/scenarios/{scenario}.json")) as f:
    df["average_n_resources"] = (
        DataFrame(json.load(f)["production_resources"]).groupby("step").size().mean()
    )

### Average number of production steps executed before aggregations

In [ ]:
from numpy import mean

# Unpack lists and take calculate mean
df["n_production_steps_aggregations"] = df["probabilities"].apply(
    lambda x: mean(
        [i for r in x for l in r["n_production_steps_aggregations"] for i in l]
    )
)

# Report results

In [ ]:
df.to_csv(f"output/{scenario}.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from pandas import concat, isna, read_csv, set_option
from scipy.stats import pearsonr

set_option("display.max_colwidth", None)

In [ ]:
scenarios = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
# scenarios = [1, 2]

df = DataFrame()
for i in scenarios:
    df = concat([df, read_csv(f"output/scenario_{i}.csv")])

df["scenario"] = df["file"].apply(lambda x: x.split("\\")[1])

df["n_steps_diff"] = df["n_steps_random"] - df["n_steps_sorted"]

In [ ]:
# If "n_steps_random" is NaN, the root cause entity does not occur in the trace
df[~isna(df["n_steps_random"])].describe()

In [ ]:
concat(
    [
        df.groupby("n_steps_random")[["kl_divergence", "n_merges"]].describe(),
        df.groupby("n_steps_sorted")[["kl_divergence", "n_merges"]].describe(),
    ]
)

In [ ]:
y_columns = {
    "n_steps_random": {
        "color": "lightblue",
        "marker": "s",
    },
    "n_steps_sorted": {
        "color": "green",
        "marker": ".",
    },
}

for x_column, x_label in [
    ("n_merges", "# merges in trace"),
    ("kl_divergence", "KL divergence uniform distribution"),
    ("split_merge_kl", "Average KL divergence for aggregations"),
    ("average_n_resources", "Average number of resources per step"),
    (
        "n_production_steps_aggregations",
        "Average number of production steps before aggregations",
    ),
]:
    fig, ax = plt.subplots(1, 1)
    for y_column, config in y_columns.items():
        df.plot.scatter(
            x=x_column,
            y=y_column,
            color=config["color"],
            marker=config["marker"],
            ax=ax,
            label=y_column,
        )

    ax.set_xlabel(x_label)
    ax.set_ylabel("# machines inspected to find root cause")
    ax.legend()
    sns.regplot(
        data=df,
        x=x_column,
        y="n_steps_diff",
        color="red",
        marker="x",
        ax=ax,
        label="n_steps_diff",
    )

    # Test if there is a positive correlation between #merges or KL divergence and the difference between n_steps_random and n_steps_sorted
    df_cor = df[~isna(df[x_column]) & ~isna(df["n_steps_diff"])]
    pcor = pearsonr(
        x=df_cor[x_column].values,
        y=(df_cor["n_steps_diff"]).values,
        alternative="greater",
    )
    print(f"{x_column} vs (n_steps_random-n_steps_sorted):", pcor)


In [ ]:
sns.scatterplot(
    data=df,
    x="kl_divergence",
    y="n_merges",
    hue="scenario",
    palette="viridis",
)
plt.xlabel("KL Divergence")
plt.ylabel("Number of merges")
plt.title("Scatter Plot of KL of path probabilities versus number of merges")
plt.legend(title="Scenario")
plt.show();

In [ ]:
sns.regplot(
    data=df[~isna(df["n_steps_random"])],
    x="split_merge_kl",
    y="n_steps_diff",
    # hue="scenario",
    # palette="viridis",
)
plt.xlabel("Mean Split-Merge KL")
plt.ylabel("KL Divergence")
plt.title("Scatter Plot of Mean Split-Merge KL vs KL of path probabilities")
plt.legend(title="Scenario")
plt.show()
# Test if there is a positive correlation between split_merge_kl and the difference between n_steps_random and n_steps_sorted
df_cor = df[~isna(df["n_steps_random"]) & ~isna(df["split_merge_kl"])]
pcor = pearsonr(
    x=df_cor["split_merge_kl"].values,
    y=(df_cor["n_steps_random"] - df_cor["n_steps_sorted"]).values,
    alternative="greater",
)
print("split_merge_kl vs (n_steps_random-n_steps_sorted):", pcor)

In [ ]:
df[
    (df["scenario"] == "scenario_10")
    & (df["split_merge_kl"] > 1)
    & ~isna(df["n_steps_random"])
]

In [ ]:
from aggregated_traces.utils.visualization import generate_graph_visualization

visualize_ekg_graph_rdf = load_rdf_graph(
    basepath.joinpath("logs/scenario_1/run_9.json")
)

# Insert DF/DP relations
visualize_ekg_graph_rdf = insert_DF_DP(visualize_ekg_graph_rdf)

# Insert fractions on the relations
visualize_ekg_graph_rdf = insert_fractions(visualize_ekg_graph_rdf)

# Create NetworkX graph
visualize_ekg_graph_nx = generate_networkx_di_graph(visualize_ekg_graph_rdf)

generate_graph_visualization(visualize_ekg_graph_nx, base_figure_path="output/check")